### ממ"ן 11 - מימוש שידור (Broadcasting) ב-PyTorch
#### מגיש - Ohad Wolfman 316552496

<div dir="rtl">


- **א.** פונקציה שמקבלת טנזורים A ו-B ומחזירה את A משודר לממדי B (מימוש של `A.expand_as(B)`). אם השידור לא אפשרי, לזרוק שגיאה. אסור לשנות את B.

**אסור:** `expand`, `expand_as`, `repeat`, `broadcast_tensors`, `broadcast_to`, `vmap`, ואסור גם לעקוף דרך numpy.

**מותר:** `squeeze`, `unsqueeze`, `cat`, `stack`, `reshape`, `reshape_as`, `clone`.

</div>

In [1]:
import torch as t

In [2]:
def my_expand_as(A, B):
    a_shape, b_shape = A.shape, B.shape

    # שלב 1: בדיקת תקינות
    if len(a_shape) > len(b_shape):
        raise ValueError(f"A has more dims than B: {tuple(a_shape)} vs {tuple(b_shape)}")
    
    for i in range(1, len(a_shape) + 1):    # ביצוע איטרציות מהסוף להתחלה
        a, b = a_shape[-i], b_shape[-i]
        if a != b and a != 1:   # אם שונים - אם לפחות איי הוא 1 אז כופלים את העמודה/המימדים בכמות המקבילה בבי
            raise ValueError(f"Cannot expand {tuple(a_shape)} to {tuple(b_shape)}: dim -{i} is {a} vs {b}")

    # שלב 2: השוואת מספר הממדים
    C = A.clone()   # הערך להחזרה עליו נבצע את ההרחבה
    while C.dim() < B.dim():
        C = C.unsqueeze(dim = 0)  # מוסיפים מימדים משמאל להשוות לגודל היעד - בי

    # שלב 3: מתיחה של כל מימד בגודל 1 לגדול המתאים
    for d in range(B.dim()):
        target = b_shape[d]     # הגודל הרצוי של המימד הנוכחי
        if C.shape[d] == 1 and target != 1:     # נבצע רק כאשר יש לאן להרחיב את המימד בסי
            if target == 0:     # הפונ' חיבור צריכה לפחות טנזור אחד, אחרת שגיאה
                C = C.narrow(dim=d, start=0, length=0)  # מתוך מימד די, קח 0 איברים החל מאינדקס 0
            else:
                C = t.cat([C] * target, dim=d)     # מדביקה את כמות העותקים זה לזה לאורך מימד די
    return C

In [3]:
# בדיקה
A = t.rand((3,1))
B = t.zeros((1,3,2))  

my_expand_as(A, B)  

tensor([[[0.3726, 0.3726],
         [0.0235, 0.0235],
         [0.8851, 0.8851]]])

<div dir="rtl">

- **ב.** פונקציה שבודקת אם שני טנזורים ניתנים לשידור יחד, ומחזירה תמיד tuple מהצורה `(True/False, shape)`. ה-shape רלוונטי רק כשהערך הראשון הוא True.
</div>

In [4]:
# בשידור דו כיווני כל טנזור יכול להימתח בממד אחר, ואילו בחד כיווני מוגדר שטנזור אחד הוא היעד הקבוע
def is_possible_to_broadcast(A,B):
    a_shape, b_shape = A.shape, B.shape
    Max = max(len(a_shape),len(b_shape))
    Min = min(len(a_shape),len(b_shape))
    result = []     # משתנה השומר את המבנה של הצורה של התוצאה

    # שלב 1: בדיקת תקינות    
    for i in range(1, Min + 1):    # ביצוע איטרציות מהסוף להתחלה
        a, b = a_shape[-i], b_shape[-i]
        if a != b and a != 1 and b != 1:   # הפעם מספיק שאחד מהם יהיה 1 ואז אותו נמתח, אבל אם הם שונים ולא 1 - שגיאה
            return (False, None)
        result.insert(0, b if a == 1 else a)    # לוקחים כל פעם את הגדול

    # שלב 2: השוואת מספר הממדים
    longer = a_shape if len(a_shape) > len(b_shape) else b_shape
    for i in range(Min + 1, Max + 1):   # נכניס את יתר המימדים מגודל המינימום עד הסוף (מקסימום)
        result.insert(0, longer[-i])

    return (True, t.Size(result))

<div dir="rtl">

- **ג.** פונקציה שמשדרת שני טנזורים יחד ומחזירה שני טנזורים חדשים (מימוש של `torch.broadcast_tensors`). מותר להשתמש בסעיפים הקודמים.
</div>

In [5]:
def my_broadcast_tensors(A, B):
    ok, shape = is_possible_to_broadcast(A, B)
    if not ok:
        raise ValueError(f"Cannot broadcast {tuple(A.shape)} and {tuple(B.shape)}")
    target = t.empty(shape)
    return my_expand_as(A, target), my_expand_as(B, target)

בדיקות

In [6]:
A, B = t.rand((3, 1)), t.rand((1, 4))
print(is_possible_to_broadcast(A, B))   # (True, torch.Size([3, 4]))
print(t.broadcast_shapes(A.shape, B.shape))     # torch.Size([3, 4])

X, Y = my_broadcast_tensors(A, B)
X2, Y2 = t.broadcast_tensors(A, B)
print(t.equal(X, X2), t.equal(Y, Y2))    # True True

(True, torch.Size([3, 4]))
torch.Size([3, 4])
True True


<div dir="rtl">

- **ד.** הרצה על דוגמאות מגוונות והשוואה לפונקציות המובנות, כדי לוודא שכל חוקי השידור מומשו נכון.
</div>

<div dir="rtl">

חילקתי את הבדיקות לשני סוגים. במקרים התקינים אני מריץ את הפונקציה שלי ואת הפונקציה המובנית, ומשווה את התוצאות. במקרים הלא תקינים אני בודק שהפונקציות שלי מזהות את הבעיה: `is_possible_to_broadcast` מחזירה False, ו-`my_expand_as` זורקת שגיאה.
בחרתי זוגות שמכסים את כל המקרים: צורות זהות, ממד חסר, 1 בצד אחד, 1 בשני הצדדים, סקלר, ממד בגודל 0, ומקרים שלא אמורים לעבוד.


</div>

In [7]:
# פונקציה ראשונה - מקרים תקינים
expand_pairs = [((1, 3), (4, 3)), ((2, 1), (2, 5)), ((3,), (2, 3)), ((), (2, 3)), ((2, 1), (2, 0))]

for a_shape, b_shape in expand_pairs:
    A, B = t.rand(a_shape), t.rand(b_shape)
    same = t.equal(my_expand_as(A, B), A.expand_as(B))
    print(a_shape, "->", b_shape, "| same answer:", same)

(1, 3) -> (4, 3) | same answer: True
(2, 1) -> (2, 5) | same answer: True
(3,) -> (2, 3) | same answer: True
() -> (2, 3) | same answer: True
(2, 1) -> (2, 0) | same answer: True


In [8]:
# can_broadcast , broadcast_tensors - מקרים תקינים
bc_pairs = [((2, 3), (2, 3)), ((3, 4), (4,)), ((3, 1), (1, 4)), ((5, 1, 4), (3, 1)), ((), (2, 3))]

for a_shape, b_shape in bc_pairs:
    A, B = t.rand(a_shape), t.rand(b_shape)
    print(a_shape, b_shape)
    print("  mine:   ", is_possible_to_broadcast(A, B))
    print("  PyTorch:", t.broadcast_shapes(A.shape, B.shape))

    X, Y = my_broadcast_tensors(A, B)
    X2, Y2 = t.broadcast_tensors(A, B)
    print("equals:", t.equal(X, X2) and t.equal(Y, Y2),"\n")

(2, 3) (2, 3)
  mine:    (True, torch.Size([2, 3]))
  PyTorch: torch.Size([2, 3])
equals: True 

(3, 4) (4,)
  mine:    (True, torch.Size([3, 4]))
  PyTorch: torch.Size([3, 4])
equals: True 

(3, 1) (1, 4)
  mine:    (True, torch.Size([3, 4]))
  PyTorch: torch.Size([3, 4])
equals: True 

(5, 1, 4) (3, 1)
  mine:    (True, torch.Size([5, 3, 4]))
  PyTorch: torch.Size([5, 3, 4])
equals: True 

() (2, 3)
  mine:    (True, torch.Size([2, 3]))
  PyTorch: torch.Size([2, 3])
equals: True 



In [9]:
# מקרים לא תקינים
bad_pairs = [((3, 4), (3,)), ((1, 2), (1, 4)), ((2, 3, 4), (3, 4))]

for a_shape, b_shape in bad_pairs:
    A, B = t.rand(a_shape), t.rand(b_shape)
    print(a_shape, b_shape, "| can_broadcast:", is_possible_to_broadcast(A, B))
    try:
        my_expand_as(A, B)
    except ValueError as e:
        print("  my_expand_as failed like we assumed:", e, "\n")

(3, 4) (3,) | can_broadcast: (False, None)
  my_expand_as failed like we assumed: A has more dims than B: (3, 4) vs (3,) 

(1, 2) (1, 4) | can_broadcast: (False, None)
  my_expand_as failed like we assumed: Cannot expand (1, 2) to (1, 4): dim -1 is 2 vs 4 

(2, 3, 4) (3, 4) | can_broadcast: (True, torch.Size([2, 3, 4]))
  my_expand_as failed like we assumed: A has more dims than B: (2, 3, 4) vs (3, 4) 

